# 🧪 Praktikum Alternatif Bab 04: Estimasi Harga Mobil Bekas Menggunakan ColumnTransformer & Scikit-Learn Pipeline
### Universitas Indraprasta PGRI (Unindra) - Laboratorium Informatika
**Mata Kuliah**: Machine Learning  
**Modul Referensi**: Bab 04  
**Fokus Topik**: Feature Engineering, Encoding Kategorikal & Scaler Terpadu (Studi Kasus: Otomotif)  

---

## 📌 Perbedaan dengan Modul Utama:
> Berbeda dari modul utama. Kita membangun full-stack Scikit-Learn Pipeline end-to-end yang memproses fitur numerik (RobustScaler) dan fitur kategorikal (OneHotEncoder) dalam satu objek estimator anti-data leakage.

---

## 🎯 Tujuan Praktikum:
1. Memahami penerapan algoritma pada studi kasus riil yang kontekstual dan mutakhir.
2. Melatih pembersihan, eksplorasi, transformasi, hingga evaluasi metrik kinerja model.
3. Menumbuhkan daya nalar komputasi mahasiswa di luar dataset standar modul kuliah.


## 1. Persiapan Environment & Import Pustaka

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


## 2. Inisialisasi & Karakterisasi Dataset Alternatif

In [ ]:
# Dataset Simulasi Mobil Bekas Indonesia
np.random.seed(55)
n_cars = 500

brands = np.random.choice(['Toyota', 'Honda', 'Daihatsu', 'Mitsubishi', 'Suzuki'], size=n_cars)
transmissions = np.random.choice(['Manual', 'Automatic'], size=n_cars, p=[0.4, 0.6])
fuel = np.random.choice(['Bensin', 'Diesel'], size=n_cars, p=[0.8, 0.2])
year = np.random.randint(2012, 2024, size=n_cars)
km = (2024 - year) * np.random.randint(8000, 25000, size=n_cars) + np.random.randint(0, 10000, size=n_cars)
engine_cc = np.random.choice([1000, 1200, 1500, 2000, 2400], size=n_cars)

base_price = (engine_cc * 120) + ((year - 2010) * 15000000) - (km * 400)
base_price += np.where(brands == 'Toyota', 20000000, 0)
base_price += np.where(transmissions == 'Automatic', 12000000, 0)
price = np.maximum(base_price + np.random.normal(0, 10000000, size=n_cars), 50000000)

df_cars = pd.DataFrame({
    'brand': brands,
    'transmission': transmissions,
    'fuel_type': fuel,
    'year': year,
    'mileage_km': km,
    'engine_cc': engine_cc,
    'price_idr': np.round(price, -5)
})

df_cars.to_csv('../../data/raw/alternatif/used_cars_indonesia.csv', index=False)
df_cars.head()


## 3. Implementasi Algoritma & Analisis Kinerja Model

In [ ]:
X = df_cars.drop('price_idr', axis=1)
y = df_cars['price_idr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_features = ['brand', 'transmission', 'fuel_type']
num_features = ['year', 'mileage_km', 'engine_cc']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

car_pricing_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

car_pricing_pipeline.fit(X_train, y_train)
y_pred = car_pricing_pipeline.predict(X_test)

print("R2 Score Pipeline:", r2_score(y_test, y_pred))
print("RMSE (Juta Rupiah):", np.sqrt(mean_squared_error(y_test, y_pred)) / 1e6)


## 4. Tugas Mandiri & Pertanyaan Diskusi Laboratorium:
1. **Analisis Sensitivitas**: Bagaimana perubahan hyperparameter mempengaruhi metrik evaluasi pada dataset ini?
2. **Kesesuaian Algoritma**: Mengapa algoritma yang digunakan cocok untuk karakteristik data di studi kasus ini?
3. **Pertimbangan Bisnis/Etika**: Jika model ini diimplementasikan di dunia nyata, apa risiko kesalahan prediksi (False Positive vs False Negative) yang paling berbahaya?
